<a href="https://colab.research.google.com/github/Dharmendra0305/PRODIGY_GA_03/blob/main/Text_Generation_with_Markov_Chains.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Phase 1: Environment Setup**

In [ ]:
# -------------------------------
# Import necessary library
# -------------------------------
from google.colab import files  # Provides utilities to upload/download files in Google Colab

# This prompts you to select the file from your computer
uploaded = files.upload()

In [ ]:
# -------------------------------
# Step 1: Open and Read the File
# -------------------------------
# 'with open' ensures the file is properly closed after reading.
# We specify UTF-8 encoding to handle special characters safely.
with open("dataset.txt", "r", encoding = "utf-8") as file:
  text = file.read()

# -------------------------------
# Step 2: Clean and Tokenize Text
# -------------------------------
# Convert all text to lowercase for consistency.
# Split the text into individual words (tokens) based on whitespace.
words = text.lower().split()

print(f"Success! Loaded {len(words)} words.")

# **Phase 2: Build the Markov Dictionary**

In [ ]:
import random  # Used for random word selection

# -------------------------------
# Step 1: Build the Markov Dictionary (Training Phase)
# -------------------------------
# The dictionary (markov_model) maps each word to a list of possible next words.
markov_model = {}

for i in range(len(words) - 1):
  current_word = words[i]       # Current word in sequence

  next_word = words[i + 1]      # Word that follows

  # If current_word not in dictionary, initialize with empty list
  if current_word not in markov_model:
    markov_model[current_word] = []

  # Append the next_word to the list of possible successors
  markov_model[current_word].append(next_word)

# Display the trained model (rulebook of word transitions)
print("The Rulebook (Model):", markov_model)

# -------------------------------
# Step 2: Generate New Text (Sampling Phase)
# -------------------------------
# Randomly pick a starting word from the model
current_word = random.choice(list(markov_model.keys()))
generated_text = [current_word]

# Generate 5 new words by following the Markov chain
for _ in range(5): # Generate 5 new words
  if current_word in markov_model:
    # Randomly select one of the possible next words

    next_word = random.choice(markov_model[current_word])
    generated_text.append(next_word)

    # Move forward in the chain
    current_word = next_word
  else:
    # Stop if no next word exists (dead end)
    break

print("\nGenerated Output:", " ".join(generated_text))


# **Phase -3: Build the Order-2 Markov Model**

In [ ]:
import random   # Used for random word selection

# -------------------------------
# Step 1: Build the Order-2 Markov Model (Training Phase)
# -------------------------------
# In an order-2 model, the "state" is a pair of consecutive words.
# Each state maps to a list of possible next words
markov_model_o2 = {}

for i in range(len(words) - 2):
  # Define the current state as a tuple of two words

  current_state = (words[i], words[i + 1])
  # The next word that follows this pair

  next_word = words[i + 2]

  # Initialize the state if not already present

  if current_state not in markov_model_o2:
    markov_model_o2[current_state] = []

  # Append the next word to the list of possible successors

  markov_model_o2[current_state].append(next_word)

# -------------------------------
# Step 2: Generate Text (Sampling Phase)
# -------------------------------
# Randomly pick a starting pair (state) from the model
current_state = random.choice(list(markov_model_o2.keys()))
generated_text = list(current_state)   # Begin with the starting pair

# Generate 50 words by following the Markov chain

for _ in range(50):
  if current_state in markov_model_o2:
    # Randomly select one of the possible next words

    next_word = random.choice(markov_model_o2[current_state])
    generated_text.append(next_word)

    # Slide the window forward:
    # (word1, word2) → (word2, next_word)
    current_state = (current_state[1], next_word)  # Move the state forward
  else:
    # Stop if we reach a state with no successors
    break

print("\nGenerated Output:", " ".join(generated_text))


# **Phase 4: Interactive Web Interface**

In [ ]:
!pip install gradio

In [ ]:
import gradio as gr # For building interactive web UI
import random       # For random word selection

# -------------------------------
# Step 1: Define Text Generation Function
# -------------------------------
def generate_text(word1, word2, num_words):
  """
    Generate text using an Order-2 Markov chain model.

    Parameters:
        word1 (str): First seed word
        word2 (str): Second seed word
        num_words (int): Desired length of output text

    Returns:
        str: Generated text or error message
    """
    # Normalize seed words (lowercase, stripped of spaces)

  current_state = (word1.lower().strip(), word2.lower().strip())

  # Check if seed words exist in the trained model
  if current_state not in markov_model_o2:
    return "Seed words not found in the training text."

# Begin generation with the seed pair
  generated = list(current_state)

  # Generate remaining words
  for _ in range(int(num_words) - 2):
    if current_state in markov_model_o2:
      # Randomly select one of the possible next words

      next_word = random.choice(markov_model_o2[current_state])
      generated.append(next_word)

       # Slide the window forward: (w1, w2) → (w2, next_word)
      current_state = (current_state[1], next_word)
    else:
      break  # Stop if no successors exist

  return " ".join(generated)

# -------------------------------
# Step 2: Build Gradio Interface
# -------------------------------
app = gr.Interface(
    fn = generate_text,     # Function to call
    inputs = [
        gr.Textbox(label = "Word 1", value = "the"),   # First seed word
        gr.Textbox(label = "Word 2", value = "cat"),   # Second seed word
        gr.Slider(minimum = 5, maximum = 100, value = 20, label = "Output Length")   # Desired length
    ],
    outputs = "text",                           # Output type
    title = "Markov Chain Text Generator",      # App title
    description = "Text Generation Project"     # Short description
)

# -------------------------------
# Step 3: Launch Application
# -------------------------------
# 'share=True' generates a public link you can share with others.
app.launch(share=True)